# 18. 연속 모니터링 임계값 캘리브레이션

held-out 정상을 검증/테스트로 분리하고, 검증 연속창에서만 임계값과 지속시간 N을 선택한 뒤 테스트를 한 번 평가합니다.

In [ ]:
## [0] 실행 환경과 테스트 전 사전 고정 규칙
import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, '..')

import glob
import librosa
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import torch

from src import config
from src.evaluate import predict_proba
from src.model import IDClassifier
from src.preprocess import sliding_windows

MACHINE = 'id_00'
MACHINE_IDX = 0
SPLIT_SEED = 42
MODEL_SEEDS = [0, 1, 2, 3, 4]
WIN_SECONDS = 10
HOP_SECONDS = 1
WIN_LEN = config.SR * WIN_SECONDS
HOP_LEN = config.SR * HOP_SECONDS
PERCENTILE_CANDIDATES = [95, 97, 99, 99.5]
N_CANDIDATES = [1, 3, 5]
TARGET_ALERTS_PER_HOUR = 1.0
OLD_N = 3
DEVICE = 'cpu'
print('테스트 전 고정 후보:', PERCENTILE_CANDIDATES, N_CANDIDATES)
print('목표 오경고 알림/시간:', TARGET_ALERTS_PER_HOUR)

In [ ]:
## [1] seed 42 held-out 정상을 짝수=검증, 홀수=테스트로 사전 분할
normalPattern = os.path.join(
    config.NOISE_DIRS['-6dB'], MACHINE, 'normal', '*.wav')
abnormalPattern = os.path.join(
    config.NOISE_DIRS['-6dB'], MACHINE, 'abnormal', '*.wav')
normalPathLST = glob.glob(normalPattern)
normalPathLST.sort()
abnormalPathLST = glob.glob(abnormalPattern)
abnormalPathLST.sort()

np.random.seed(SPLIT_SEED)
indexNP = np.random.permutation(len(normalPathLST))
TRAIN_NUM = int(len(normalPathLST) * config.TRAIN_RATIO)
heldoutIndexNP = indexNP[TRAIN_NUM:]
heldoutNormalPathLST = []
for heldout_idx in heldoutIndexNP:
    heldoutNormalPathLST.append(normalPathLST[heldout_idx])
heldoutNormalPathLST.sort()

validationNormalPathLST = []
testNormalPathLST = []
for heldout_idx in range(len(heldoutNormalPathLST)):
    if heldout_idx % 2 == 0:
        validationNormalPathLST.append(heldoutNormalPathLST[heldout_idx])
    else:
        testNormalPathLST.append(heldoutNormalPathLST[heldout_idx])

print('검증 정상 파일', len(validationNormalPathLST), '개:')
print([os.path.basename(path) for path in validationNormalPathLST])
print('테스트 정상 파일', len(testNormalPathLST), '개:')
print([os.path.basename(path) for path in testNormalPathLST])
print('테스트 이상 파일:', len(abnormalPathLST), '개')

In [ ]:
## [2] 배포 앙상블과 기존 임계값 로드
modelLST = []
for seed in MODEL_SEEDS:
    model = IDClassifier()
    modelPath = os.path.join(config.MODEL_DIR, f'idclf_{seed}.pth')
    model.load_state_dict(torch.load(
        modelPath, map_location=DEVICE, weights_only=True))
    model.eval()
    modelLST.append(model)
MIN, MAX = np.load(os.path.join(config.MODEL_DIR, 'clf_minmax.npy'))
thresholdNP = np.load(os.path.join(
    config.MODEL_DIR, 'clf_thresholds.npy'))
OLD_THRESHOLD = thresholdNP[MACHINE_IDX]
warmupNP = np.zeros((1, 20032), dtype=np.float32)
for model in modelLST:
    predict_proba(model, warmupNP, DEVICE)
print('기존 train-90% 임계값:', OLD_THRESHOLD)

## 검증셋 캘리브레이션

아래 셀은 검증 정상만 읽습니다. 테스트 파일의 점수는 아직 계산하지 않습니다.

In [ ]:
## [3] 검증 정상 107개를 연결하고 연속창 점수 계산
validationSignalLST = []
for path in validationNormalPathLST:
    yNP, sr = librosa.load(path, sr=config.SR)
    if len(yNP) != WIN_LEN:
        raise ValueError(f'검증 WAV 길이 오류: {path}')
    validationSignalLST.append(yNP)
validationSignalNP = np.concatenate(validationSignalLST)
validationWindowLST = sliding_windows(
    validationSignalNP, WIN_LEN, HOP_LEN)
validationScoreLST = []
for windowNP in validationWindowLST:
    melNP = librosa.feature.melspectrogram(
        y=windowNP, sr=config.SR, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
    mel_dbNP = librosa.power_to_db(melNP)
    xNP = ((mel_dbNP - MIN) / (MAX - MIN)).reshape(1, -1)
    scoreSUM = 0.0
    for model in modelLST:
        probaNP = predict_proba(model, xNP, DEVICE)
        scoreSUM = scoreSUM + (1 - probaNP[0, MACHINE_IDX])
    validationScoreLST.append(scoreSUM / len(modelLST))
validationScoreNP = np.array(validationScoreLST)
VALIDATION_HOURS = len(validationSignalNP) / config.SR / 3600
print('검증 연속창:', len(validationScoreNP), '개')
print('검증 시간:', VALIDATION_HOURS, '시간')

In [ ]:
## [4] 사전 후보를 검증 정상에만 적용해 임계값·N 동결
candidateResultLST = []
SELECTED_THRESHOLD = np.nan
SELECTED_PERCENTILE = np.nan
SELECTED_N = np.nan
selectionFound = False
fallbackResult = None

print('퍼센타일 | N | 창 FPR | 알림 건수 | 건/시간 | 건/8h')
print('-' * 72)
for required_count in N_CANDIDATES:
    for percentile in PERCENTILE_CANDIDATES:
        candidateThreshold = np.percentile(
            validationScoreNP, percentile)
        exceedFlagNP = validationScoreNP > candidateThreshold
        consecutive_count = 0
        warningFlagNP = np.zeros(len(validationScoreNP), dtype=bool)
        for window_idx in range(len(validationScoreNP)):
            if exceedFlagNP[window_idx]:
                consecutive_count = consecutive_count + 1
            else:
                consecutive_count = 0
            if consecutive_count >= required_count:
                warningFlagNP[window_idx] = True
        alertTriggerNP = np.zeros(len(validationScoreNP), dtype=bool)
        for window_idx in range(len(warningFlagNP)):
            previous_warning = False
            if window_idx > 0:
                previous_warning = warningFlagNP[window_idx - 1]
            if warningFlagNP[window_idx] and not previous_warning:
                alertTriggerNP[window_idx] = True
        WINDOW_FPR = exceedFlagNP.mean()
        ALERT_COUNT = int(np.sum(alertTriggerNP))
        ALERTS_PER_HOUR = ALERT_COUNT / VALIDATION_HOURS
        result = {
            'percentile': percentile, 'threshold': candidateThreshold,
            'N': required_count, 'windowFPR': WINDOW_FPR,
            'alertCount': ALERT_COUNT,
            'alertsPerHour': ALERTS_PER_HOUR}
        candidateResultLST.append(result)
        print(
            f'{percentile} | {required_count} | {WINDOW_FPR:.4f} | '
            f'{ALERT_COUNT} | {ALERTS_PER_HOUR:.3f} | '
            f'{ALERTS_PER_HOUR * 8:.2f}')

        if not selectionFound and ALERTS_PER_HOUR <= TARGET_ALERTS_PER_HOUR:
            SELECTED_THRESHOLD = candidateThreshold
            SELECTED_PERCENTILE = percentile
            SELECTED_N = required_count
            selectionFound = True

        if fallbackResult is None:
            fallbackResult = result
        elif ALERTS_PER_HOUR < fallbackResult['alertsPerHour']:
            fallbackResult = result

if not selectionFound:
    SELECTED_THRESHOLD = fallbackResult['threshold']
    SELECTED_PERCENTILE = fallbackResult['percentile']
    SELECTED_N = fallbackResult['N']
    print('목표 미충족: 검증 오경고 최소 후보를 fallback으로 선택')

SELECTED_N = int(SELECTED_N)
print('동결 임계값:', SELECTED_THRESHOLD)
print('동결 퍼센타일/N:', SELECTED_PERCENTILE, SELECTED_N)

## 🔒 여기서부터 최종 테스트 — 설정 변경 금지

위 검증셋에서 동결한 `SELECTED_THRESHOLD`와 `SELECTED_N`을 그대로 사용합니다. 아래 결과를 본 뒤 재튜닝하지 않습니다.

In [ ]:
## [5] 테스트 1회 — 정상 107개·이상 356개 클립 수준 평가
testNormalMelLST = []
for path in testNormalPathLST:
    yNP, sr = librosa.load(path, sr=config.SR)
    melNP = librosa.feature.melspectrogram(
        y=yNP, sr=config.SR, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
    testNormalMelLST.append(librosa.power_to_db(melNP))
testAbnormalMelLST = []
for path in abnormalPathLST:
    yNP, sr = librosa.load(path, sr=config.SR)
    melNP = librosa.feature.melspectrogram(
        y=yNP, sr=config.SR, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
    testAbnormalMelLST.append(librosa.power_to_db(melNP))
testNormalNP = (np.array(testNormalMelLST) - MIN) / (MAX - MIN)
testAbnormalNP = (np.array(testAbnormalMelLST) - MIN) / (MAX - MIN)
normalScoreSUM = np.zeros(len(testNormalNP))
abnormalScoreSUM = np.zeros(len(testAbnormalNP))
for model in modelLST:
    normalProbaNP = predict_proba(model, testNormalNP, DEVICE)
    abnormalProbaNP = predict_proba(model, testAbnormalNP, DEVICE)
    normalScoreSUM = normalScoreSUM + (1 - normalProbaNP[:, MACHINE_IDX])
    abnormalScoreSUM = abnormalScoreSUM + (1 - abnormalProbaNP[:, MACHINE_IDX])
testNormalScoreNP = normalScoreSUM / len(modelLST)
testAbnormalScoreNP = abnormalScoreSUM / len(modelLST)
clipResultDCT = {}
for threshold_name, threshold in [
        ('기존', OLD_THRESHOLD), ('새 임계값', SELECTED_THRESHOLD)]:
    NORMAL_FPR = np.mean(testNormalScoreNP > threshold)
    ABNORMAL_TPR = np.mean(testAbnormalScoreNP > threshold)
    clipResultDCT[threshold_name] = {
        'normalFPR': NORMAL_FPR, 'abnormalTPR': ABNORMAL_TPR}
    print(
        f'{threshold_name}: 정상 오경고율 {NORMAL_FPR:.4f} / '
        f'이상 탐지율 {ABNORMAL_TPR:.4f}')

In [ ]:
## [6] 테스트 1회 — 정상 연속 대조군과 정상→이상→정상 시나리오
testNormalSignalLST = []
for path in testNormalPathLST:
    yNP, sr = librosa.load(path, sr=config.SR)
    testNormalSignalLST.append(yNP)
testNormalSignalNP = np.concatenate(testNormalSignalLST)
testNormalWindowLST = sliding_windows(
    testNormalSignalNP, WIN_LEN, HOP_LEN)
testNormalWindowScoreLST = []
for windowNP in testNormalWindowLST:
    melNP = librosa.feature.melspectrogram(
        y=windowNP, sr=config.SR, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
    mel_dbNP = librosa.power_to_db(melNP)
    xNP = ((mel_dbNP - MIN) / (MAX - MIN)).reshape(1, -1)
    scoreSUM = 0.0
    for model in modelLST:
        probaNP = predict_proba(model, xNP, DEVICE)
        scoreSUM = scoreSUM + (1 - probaNP[0, MACHINE_IDX])
    testNormalWindowScoreLST.append(scoreSUM / len(modelLST))
testNormalWindowScoreNP = np.array(testNormalWindowScoreLST)
TEST_NORMAL_HOURS = len(testNormalSignalNP) / config.SR / 3600

# 대표 test-only 혼합 시나리오 파일도 결과 전에 정렬 첫 순서로 고정했다.
mixedNormalPathLST = testNormalPathLST[:3]
mixedAbnormalPath = abnormalPathLST[0]
mixedSignalLST = []
for path in mixedNormalPathLST[:2]:
    yNP, sr = librosa.load(path, sr=config.SR)
    mixedSignalLST.append(yNP)
mixedAbnormalNP, sr = librosa.load(mixedAbnormalPath, sr=config.SR)
mixedSignalLST.append(mixedAbnormalNP)
lastNormalNP, sr = librosa.load(mixedNormalPathLST[2], sr=config.SR)
mixedSignalLST.append(lastNormalNP)
mixedSignalNP = np.concatenate(mixedSignalLST)
mixedWindowLST = sliding_windows(mixedSignalNP, WIN_LEN, HOP_LEN)
mixedScoreLST = []
mixedEndTimeLST = []
for window_idx in range(len(mixedWindowLST)):
    windowNP = mixedWindowLST[window_idx]
    melNP = librosa.feature.melspectrogram(
        y=windowNP, sr=config.SR, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH)
    mel_dbNP = librosa.power_to_db(melNP)
    xNP = ((mel_dbNP - MIN) / (MAX - MIN)).reshape(1, -1)
    scoreSUM = 0.0
    for model in modelLST:
        probaNP = predict_proba(model, xNP, DEVICE)
        scoreSUM = scoreSUM + (1 - probaNP[0, MACHINE_IDX])
    mixedScoreLST.append(scoreSUM / len(modelLST))
    mixedEndTimeLST.append(
        (window_idx * HOP_LEN + WIN_LEN) / config.SR)
mixedScoreNP = np.array(mixedScoreLST)
mixedEndTimeNP = np.array(mixedEndTimeLST)
mixedAbnormalFractionLST = []
for window_idx in range(len(mixedEndTimeNP)):
    window_start = window_idx * HOP_SECONDS
    window_end = window_start + WIN_SECONDS
    overlap_start = max(window_start, 20.0)
    overlap_end = min(window_end, 30.0)
    mixedAbnormalFractionLST.append(
        max(0.0, overlap_end - overlap_start) / WIN_SECONDS)
mixedAbnormalFractionNP = np.array(mixedAbnormalFractionLST)

monitorConfigLST = [
    ('기존 임계값, N=3', OLD_THRESHOLD, OLD_N),
    ('새 임계값, N=1', SELECTED_THRESHOLD, 1),
    (f'새 임계값, N={SELECTED_N}', SELECTED_THRESHOLD, SELECTED_N)]
continuousResultDCT = {}
for config_name, threshold, required_count in monitorConfigLST:
    normalExceedNP = testNormalWindowScoreNP > threshold
    consecutive_count = 0
    normalWarningNP = np.zeros(len(normalExceedNP), dtype=bool)
    for window_idx in range(len(normalExceedNP)):
        if normalExceedNP[window_idx]:
            consecutive_count = consecutive_count + 1
        else:
            consecutive_count = 0
        if consecutive_count >= required_count:
            normalWarningNP[window_idx] = True
    normalTriggerNP = np.zeros(len(normalWarningNP), dtype=bool)
    for window_idx in range(len(normalWarningNP)):
        previous_warning = False
        if window_idx > 0:
            previous_warning = normalWarningNP[window_idx - 1]
        if normalWarningNP[window_idx] and not previous_warning:
            normalTriggerNP[window_idx] = True

    mixedExceedNP = mixedScoreNP > threshold
    consecutive_count = 0
    mixedWarningNP = np.zeros(len(mixedExceedNP), dtype=bool)
    for window_idx in range(len(mixedExceedNP)):
        if mixedExceedNP[window_idx]:
            consecutive_count = consecutive_count + 1
        else:
            consecutive_count = 0
        if consecutive_count >= required_count:
            mixedWarningNP[window_idx] = True
    mixedTriggerNP = np.zeros(len(mixedWarningNP), dtype=bool)
    for window_idx in range(len(mixedWarningNP)):
        previous_warning = False
        if window_idx > 0:
            previous_warning = mixedWarningNP[window_idx - 1]
        if mixedWarningNP[window_idx] and not previous_warning:
            mixedTriggerNP[window_idx] = True
    FALSE_ALERTS = int(np.sum(
        mixedTriggerNP & (mixedAbnormalFractionNP == 0)))
    FIRST_VALID_ALERT = np.nan
    for window_idx in range(len(mixedTriggerNP)):
        if mixedTriggerNP[window_idx] and mixedAbnormalFractionNP[window_idx] > 0:
            FIRST_VALID_ALERT = mixedEndTimeNP[window_idx]
            break
    DETECTION_DELAY = FIRST_VALID_ALERT - 20.0
    if np.isnan(FIRST_VALID_ALERT):
        DETECTION_DELAY = np.nan
    continuousResultDCT[config_name] = {
        'normalWindowFPR': normalExceedNP.mean(),
        'normalAlertCount': int(np.sum(normalTriggerNP)),
        'normalAlertsPerHour': int(np.sum(normalTriggerNP)) / TEST_NORMAL_HOURS,
        'mixedTriggerNP': mixedTriggerNP,
        'mixedFalseAlerts': FALSE_ALERTS,
        'firstValidAlert': FIRST_VALID_ALERT,
        'detectionDelay': DETECTION_DELAY}
    print(config_name, continuousResultDCT[config_name])

In [ ]:
## [7] untouched test 시나리오의 튜닝 전/후 정직 비교 그래프
plotConfigLST = [
    ('튜닝 전: train-90%, N=3', '기존 임계값, N=3', OLD_THRESHOLD),
    (f'튜닝 후: val-{SELECTED_PERCENTILE}%, N={SELECTED_N}',
     f'새 임계값, N={SELECTED_N}', SELECTED_THRESHOLD)]
fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=True, sharey=True)
for plot_idx in range(len(plotConfigLST)):
    title, result_key, activeThreshold = plotConfigLST[plot_idx]
    ax = axes[plot_idx]
    result = continuousResultDCT[result_key]
    triggerNP = result['mixedTriggerNP']
    falseMaskNP = triggerNP & (mixedAbnormalFractionNP == 0)
    validMaskNP = triggerNP & (mixedAbnormalFractionNP > 0)
    ax.plot(mixedEndTimeNP, mixedScoreNP, marker='o', label='앙상블 이상점수')
    ax.axhline(OLD_THRESHOLD, color='tab:red', linestyle='--',
               label=f'기존 임계값 {OLD_THRESHOLD:.3f}')
    ax.axhline(SELECTED_THRESHOLD, color='tab:green', linestyle='--',
               label=f'새 임계값 {SELECTED_THRESHOLD:.3f}')
    ax.axvspan(20, 30, color='tab:red', alpha=0.12, label='실제 이상 구간')
    for seam_time in [10, 20, 30]:
        ax.axvline(seam_time, color='gray', linestyle=':', alpha=0.6)
    ax.scatter(mixedEndTimeNP[falseMaskNP], mixedScoreNP[falseMaskNP],
               marker='X', s=150, color='black', zorder=5,
               label='정상 오경고 알림')
    ax.scatter(mixedEndTimeNP[validMaskNP], mixedScoreNP[validMaskNP],
               marker='*', s=220, color='crimson', edgecolor='black',
               zorder=5, label='이상 유효 알림')
    ax.set_title(title)
    ax.set_ylabel('앙상블 이상점수')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right')
axes[1].set_xlabel('창 종료시간(초) — untouched 테스트 판정 시점')
axes[1].set_xlim(9.5, 40.5)
plt.suptitle('검증 캘리브레이션 전/후 연속 모니터링 비교', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/sliding_window_before_after.png',
            dpi=120, bbox_inches='tight')
plt.show()

## 결과 요약

- 검증 연속창 1,061개(약 17.8분)에서 후보를 비교해 **99퍼센타일 임계값 0.864715, N=3**을 테스트 전에 동결했습니다. 검증 오경고 알림은 0건이었습니다.
- 클립 테스트: 정상 오경고율은 기존 33.64%에서 **0.93%**로 감소했지만, 이상 356개 탐지율도 100%에서 **72.47%**로 하락했습니다.
- 정상 연속 테스트: 기존 N=3은 창 FPR 36.19%, 알림 37건(124.49건/시간); 새 N=3은 창 FPR **0.47%**, 알림 1건(**3.36건/시간**)이었습니다.
- test-only 혼합 시나리오: 기존 규칙은 정상 오경고가 먼저 시작돼 별도 유효 알림이 없었습니다. 새 임계값 N=1은 25초(5초 지연), 선택 N=3은 27초(7초 지연)에 유효 알림이 발생했고 정상 오경고는 0건이었습니다.
- 테스트 결과를 확인한 뒤 임계값·N을 다시 조정하지 않았습니다. 오경고는 크게 감소했지만 테스트의 시간당 목표 1건은 충족하지 못했고 탐지율 손실이 커서, 현재 캘리브레이션을 최종 배포 규칙으로 확정할 수 없습니다.
- 검증·테스트가 각각 약 18분이고 MIMII 토막 파일 이음새를 포함하므로, 실제 프레스 장시간 연속 정상 데이터로 다시 검증해야 합니다.